# Predictions from Baryon Power Spectrum

Because the probability of conversion is a function of $m_\gamma^2$, which is in turn a function of the baryon power spectrum, we can predict the power spectrum of the probability of conversion (as well as the global signal) from $P_{bb}$, which we can simulate from `CLASS`. Of course, this assumes that we can neglect perturbations in the free electron fraction, which is only true about $z\approx 20$. 

To do this, we first import a power spectrum $P_{\rm bb}$ from `CLASS' and compute
$$ \begin{align} \sigma_{\rm b}^2(z) &= \int \frac{d^3\vec{k}}{(2\pi)^3}P_{\rm bb}(k,z).
\end{align}$$


In [ ]:
import os, sys
import pickle
sys.path.append("../")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec
from matplotlib import ticker
from scipy.optimize import fsolve
import seaborn as sns
import numpy as np
from scipy.integrate import cumulative_trapezoid
from tqdm import *
from scipy.interpolate import interp1d
grf_path = "/home/bakerem/dark-photons-perturbations"
sys.path.append(grf_path)
from grf.grf import PerturbedProbability, FIRAS
from grf.pk_interp import PowerSpectrumGridInterpolator
from grf.units import *

from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%matplotlib inline
%load_ext autoreload
%autoreload 2
# Load plot settings

from plot_params import params
pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']
# Where to save plots
plots_dir = "../paper/draft-formalism/plots/"
z_compute_ary = np.logspace(-3, 3, 500)
k_ary = np.logspace(-4, np.log10(1e3), 500)

log_pk_lin_baryon_grid_ary = np.zeros((len(z_compute_ary), len(k_ary)))

for i_z, z_compute in enumerate(z_compute_ary):
    ary = np.load(grf_path+"/data/pk_arys/p_k_k_max_5000_z_" + str(i_z) + ".npz")
    interp_lin_baryon = interp1d(np.log10(ary['k_ary']), np.log10(ary['Pk_b_ary']), bounds_error=False, fill_value="extrapolate")
log_pspec = PowerSpectrumGridInterpolator("lin_baryon")

In [ ]:

mA_list = [3e-14, 8e-14]  # eV
labels = [r"$m_{A'}=3\times 10^{-14}$ eV", r"$m_{A'}=8\times 10^{-14}$ eV"]
for i, mA in enumerate(mA_list):
    prob = PerturbedProbability(log_pspec)

    # Non-linear matter power spectrum. 
    pspec_lin_baryon = PowerSpectrumGridInterpolator("lin_baryon")

    # Class containing results with linear baryon spectrum. 
    prob = FIRAS(pspec_lin_baryon)

    one_plus_delta_bound = 1e2  # Fiducial bound
    omega_ary = 115e6 * 6.58e-16 * 2 * np.pi

    epsilon = 1e-6
    test_z_array = np.geomspace(0.05, 100, 1000)
    dPdz_ary_1 = epsilon**2 * prob._dP_dz(z_ary=test_z_array, m_Ap=mA * eV, k_min=1e-3, k_max=0.25, omega=omega_ary, pdf="lognormal", one_plus_delta_bound=one_plus_delta_bound, return_pdf=True)[0][0]
    # dPdz_ary_2 = epsilon**2 * prob._dP_dz(z_ary=test_z_array, m_Ap=1e-14 * eV, k_min=1e-3, k_max=0.25, omega=omega_ary, pdf="lognormal", one_plus_delta_bound=one_plus_delta_bound, return_pdf=True)[0][0]

    plt.loglog(test_z_array, dPdz_ary_1, label=labels[i])
    # plt.plot(test_z_array, dPdz_ary_2, label=r"$m_{A'}=3\times 10^{-14}$ eV")
    # plt.xscale("log")
    # plt.yscale("log")
plt.ylim(1e-7, 1e-2)
plt.xlabel(r"$z$")
plt.ylabel(r"$d\langle P_{\gamma \to A'}\rangle/dz$")
plt.legend(loc="upper right")
plt.savefig("plots/dPdz_analytic.pdf", bbox_inches='tight')
